# Baseline Notebook: Self-Consistency Prompting

Notebook này là bản **baseline đơn giản** cho kỹ thuật self-consistency prompting. Ý tưởng ở đây không dùng embedding hay semantic clustering. Ta chỉ:
- gọi model nhiều lần với cùng một prompt,
- ép model trả về nhãn cuối cùng theo format ổn định,
- đếm phiếu để chọn đáp án xuất hiện nhiều nhất.

Vì bài toán là phân loại email thành `IMPORTANT` hoặc `NOT IMPORTANT`, cách majority vote trực tiếp trên nhãn là đủ gọn và rất dễ giải thích.

## Self-Consistency bản baseline hoạt động thế nào?

Pipeline của notebook này gồm 3 bước:
1. chạy một lần để xem kết quả baseline thông thường,
2. chạy nhiều lần với `temperature > 0` để tạo nhiều mẫu suy luận,
3. trích nhãn cuối cùng từ từng mẫu rồi majority vote.

Điểm quan trọng là ta **không vote trên toàn bộ câu trả lời**, mà vote trên **nhãn cuối cùng**. Nhờ đó code ngắn, ổn định, và phù hợp với các tác vụ classification có tập nhãn rõ ràng.

## Chuẩn bị môi trường

Điều kiện để notebook chạy được:
- LM Studio đang chạy local server tại `http://localhost:1234`
- một chat model đang được load trong LM Studio
- môi trường Python đã có thư viện `openai`

Nếu chưa có thư viện, chạy cell cài đặt bên dưới một lần.

In [ ]:
# Bỏ comment nếu môi trường của bạn chưa có thư viện cần thiết
# %pip install -q openai


In [ ]:
from collections import Counter
from openai import OpenAI

LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
API_KEY = "lm-studio"
CHAT_MODEL = "local-model"

TEMPERATURE = 0.8
NUM_SAMPLES = 5
MAX_TOKENS = 512

def build_client(base_url=LM_STUDIO_BASE_URL):
    return OpenAI(base_url=base_url, api_key=API_KEY)

client = build_client()
print(f"Đã kết nối LM Studio qua {LM_STUDIO_BASE_URL}")


## Thiết kế prompt

Bài toán mẫu là một email có dấu hiệu khai thác lỗ hổng bảo mật. Để bước vote ổn định hơn, ta yêu cầu model kết thúc bằng một dòng nhãn chuẩn hóa:

- `Final label: IMPORTANT`
- hoặc `Final label: NOT IMPORTANT`

Cách ép output format như vậy giúp phần hậu xử lý rất gọn.

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful security triage assistant. "
    "Reason clearly and end with one exact label line."
)

EMAIL_PROMPT = """Read the email below and classify it as IMPORTANT or NOT IMPORTANT.
Explain briefly, then end with exactly one of these two lines:
- Final label: IMPORTANT
- Final label: NOT IMPORTANT

EMAIL:

Hi,

I have seen you use Wordpress for your website. A great open source content management system. I have used it in the past too. It comes with lots of great user plugins. And it's pretty easy to set up.

I did notice a bug in the contact form, which happens when you select the name field. See the attached screenshot of me entering text in the name field. Notice the JavaScript alert box that I inv0k3d.

But for the rest it's a great website. I enjoy reading it. Feel free to leave the bug in the website, because it gives me more interesting things to read.

Cheers,

Harry the Hacker.

Let's think step by step and explain why."""

def generate_completion(prompt, temperature=TEMPERATURE):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content.strip()

def collect_reasoning_paths(prompt, num_samples=NUM_SAMPLES, temperature=TEMPERATURE):
    samples = []
    for sample_id in range(1, num_samples + 1):
        answer = generate_completion(prompt, temperature=temperature)
        samples.append(answer)

        print(f"--- Mẫu suy luận {sample_id} ---")
        print(answer)
        print("-" * 80)

    return samples


## 1. Baseline: chỉ gọi model một lần

Đây là cách dùng prompt bình thường: model chạy một lần và ta lấy luôn kết quả đó.

In [ ]:
baseline_answer = generate_completion(EMAIL_PROMPT, temperature=TEMPERATURE)

print("=== Baseline answer ===")
print(baseline_answer)


## 2. Chạy self-consistency

Bây giờ ta gọi model nhiều lần với cùng prompt để lấy nhiều đường suy luận độc lập. Vì đây là tác vụ phân loại, ta không cần gom cụm theo nghĩa; chỉ cần rút nhãn cuối cùng từ từng mẫu là đủ.

In [ ]:
responses = collect_reasoning_paths(EMAIL_PROMPT)


## 3. Majority vote trên nhãn cuối cùng

Phần này là cốt lõi của bản baseline:
- đọc từng response,
- trích nhãn `IMPORTANT` hoặc `NOT IMPORTANT`,
- đếm phiếu,
- chọn nhãn nhiều phiếu nhất làm kết quả cuối.

Cách này rất hợp khi output space nhỏ và rõ ràng, ví dụ classification, routing, spam detection, safety triage, hoặc ticket priority.

In [ ]:
def extract_label(text):
    upper_text = text.upper()

    if "FINAL LABEL: NOT IMPORTANT" in upper_text:
        return "NOT IMPORTANT"
    if "FINAL LABEL: IMPORTANT" in upper_text:
        return "IMPORTANT"

    # Fallback nếu model không tuân thủ format hoàn toàn
    if "NOT IMPORTANT" in upper_text:
        return "NOT IMPORTANT"
    if "IMPORTANT" in upper_text:
        return "IMPORTANT"
    return "UNKNOWN"

def majority_vote(texts):
    labels = [extract_label(text) for text in texts]
    vote_counts = Counter(labels)
    winning_label, winning_votes = vote_counts.most_common(1)[0]
    confidence = winning_votes / len(texts)

    print("=== Nhãn trích xuất từ từng mẫu ===")
    for idx, label in enumerate(labels, start=1):
        print(f"Mẫu {idx}: {label}")
    print()

    print("=== Kết quả biểu quyết ===")
    print(f"Chi tiết phiếu bầu: {dict(vote_counts)}")
    print(f"Nhãn chiến thắng: {winning_label} ({winning_votes}/{len(texts)} phiếu)")
    print(f"Độ tự tin ước lượng: {confidence:.0%}")

    return winning_label, confidence

final_label, confidence = majority_vote(responses)


## 4. Ghi chú kỹ thuật

Bản baseline này có ưu điểm là rất gọn và dễ triển khai, nhưng cũng có giới hạn:
- nó phù hợp nhất khi model trả về tập nhãn nhỏ, rõ ràng,
- nó phụ thuộc vào việc model giữ format ổn định,
- nếu output là câu tự do dài hoặc có nhiều cách diễn đạt khác nhau, exact-label voting sẽ yếu hơn semantic grouping bằng embedding.

Nói ngắn gọn: notebook này phù hợp cho **classification-style self-consistency**, còn notebook trước phù hợp cho **reasoning-style self-consistency với câu trả lời tự do**.